# 12 · Ablation "có / không OCR" trên ViTextVQA test2000 (W8, bước 6)

**Vấn đề:** Đề cương bắt buộc ablation "with / without OCR-enhanced prompting". Cấu hình OCR đã được **chọn và đóng băng trên dev300** (notebook 11, theo luật đặt trước): **VietOCR · template v1 · max_chars 600 · min_conf 0**. Notebook này chạy cấu hình đó **một lần** trên test2000, cho cả hai mô hình đã có:

| Run | Mô hình | OCR | Ghi chú |
|---|---|---|---|
| B1 | zero-shot | không | đã có (W07) |
| **B1 + OCR** | zero-shot | VietOCR v1 | **chạy mới** |
| B2 | QLoRA 1 epoch (train full ViTextVQA, không OCR) | không | đã có (W07) |
| **B2 + OCR** | như trên | VietOCR v1 | **chạy mới** — chỉ chèn OCR lúc suy luận, B2 **chưa từng thấy** dòng OCR khi train (lệch train/test) → dòng chẩn đoán, không phải kết quả cải tiến |

**Luật kết luận (ghi trong `docs/W08/README.md` trước khi chạy):** OCR "cải thiện" một mô hình nếu EM tăng **và** McNemar exact p < 0,05; ANLS báo cáo kèm CI95 bootstrap ghép cặp. Trên dev300, OCR **không** giúp B1 (EM 0,3267 → 0,3267, p = 1). Kết quả âm trên test vẫn là một dòng ablation hợp lệ.

**Runtime: L4.** Khoảng 70–80 phút, chạy được bằng **Run all**.

## 1. Thiết lập môi trường

Giống notebook 02/04/08/09/11 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (có notebook 12), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('notebooks/12_ocr_ablation_test2000_qwen25vl.ipynb'), 'Code cũ -> git pull chưa lấy được commit bước 6'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B1/B2). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Cấu hình OCR đã đóng băng + kiểm tra trước khi tốn GPU

- Đọc cấu hình OCR từ `experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json` (kết quả notebook 11, đã lên git), **không gõ tay**.
- Adapter B2 phải đúng **sha256** đã chấm ở W07, để khẳng định cùng một B2.
- OCR cache VietOCR phải **đủ cả 1.486 ảnh** của test2000 (Phần B notebook 10).
- Hai run W07 (B1, B2 trên test2000) phải có trên Drive để so ghép cặp.

*Kỳ vọng: `paddledet_vietocr | v1 | 600 | 0.0`, `adapter sha256 OK`, `1486/1486 ảnh`, hai dòng `✓ W07_…`, `SELFTEST OK`.*

In [ ]:
import json, yaml
from src.data.ocr import default_cache_path, load_ocr_cache, images_of
from src.eval.run_baseline import prepare_images, adapter_fingerprint, DATASETS

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
TEST = f"{DRIVE_ROOT}/{DATASETS[DS]['data_rel']}"
ADAPTER = f'{DRIVE_ROOT}/adapters/W07_B2_qlora_r16_qwen25vl_vitextvqa_ep1_bf16_L4_seed42/epoch_1'
ADAPTER_SHA256 = '2e7b17d18b32b0f248b130dab4717b9b408b1da68e4fea5c3f5d6baaf9dcf863'   # adapter đã chấm ở W07
OUT_DIR = f'{EXP_ROOT}/W08_ocr_ablation_test2000_L4_seed42'

sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
assert (BACKEND, TPL) == ('paddledet_vietocr', 'v1'), 'Cấu hình OCR khác bản đã chốt ở bước 5'

RUNS = {
    'B1':     {'run': 'W07_B1_zeroshot_qwen25vl_vitextvqa_test2000_bf16_L4_seed42', 'new': False},
    'B1+OCR': {'run': 'W08_B1ocr_vietocr_v1_qwen25vl_vitextvqa_test2000_bf16_L4_seed42', 'new': True, 'adapter': None},
    'B2':     {'run': 'W07_B2_eval_qlora_r16_ep1_vitextvqa_test2000_bf16_L4_seed42', 'new': False},
    'B2+OCR': {'run': 'W08_B2ocr_vietocr_v1_qlora_r16_ep1_vitextvqa_test2000_bf16_L4_seed42', 'new': True, 'adapter': ADAPTER},
}

cfg = yaml.safe_load(open('configs/qwen_lora.yaml', encoding='utf-8'))
print(cfg['model']['torch_dtype'], '|', cfg['model']['max_pixels'], '|', cfg['quantization']['bnb_4bit_quant_type'])
fp = adapter_fingerprint(ADAPTER)
assert fp['weights_sha256'] == ADAPTER_SHA256, 'Adapter B2 khác bản W07!'
print('adapter sha256 OK', fp['weights_mb'], 'MB')
IMG = prepare_images(DS, DRIVE_ROOT)
need = set(images_of([TEST], IMG))
c = load_ocr_cache(default_cache_path(DRIVE_ROOT, DS, BACKEND))
have = len(need & set(c['images']))
print(f'{BACKEND}: {have}/{len(need)} ảnh test2000')
assert have == len(need), 'OCR cache thiếu ảnh test2000 -> chạy notebook 10 Phần B'
for k, r in RUNS.items():
    if not r['new']:
        assert os.path.exists(f"{EXP_ROOT}/{r['run']}/predictions.json"), f"Thiếu run W07 {r['run']}"
        print('✓', r['run'])
!python -m src.eval.run_baseline --selftest 2>&1 | tail -1

## 3. Chạy B1 + OCR và B2 + OCR trên test2000 (≈ 65–75 phút)

Cùng script `run_baseline.py`, cùng test2000 (mặc định của `vitextvqa_official`), cùng cấu hình chốt W07. Chỉ thêm `--mode ocr` cùng các thiết lập OCR vừa đọc. B2 + OCR gắn **đúng adapter B2** đã kiểm sha256. Run nào đã có `metrics.json` thì bỏ qua; Colab ngắt giữa chừng thì chạy lại, các lô 500 câu đã xong sẽ được giữ.

*Kỳ vọng: mỗi run in dòng `OCR: … % mẫu có chữ`, 4 lô, rồi `== KẾT QUẢ: EM=…`.*

In [ ]:
for k, r in RUNS.items():
    if not r['new']:
        continue
    run_name = r['run']
    if os.path.exists(f'{EXP_ROOT}/{run_name}/metrics.json'):
        print(f'⏭ {k}: đã có -> bỏ qua'); continue
    print(f'\n▶ {k}: {run_name}')
    if r['adapter']:
        !python -u -m src.eval.run_baseline --dataset {DS} --run_name {run_name} --mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC} --adapter {ADAPTER}
    else:
        !python -u -m src.eval.run_baseline --dataset {DS} --run_name {run_name} --mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}

## 4. Kết quả theo luật

- **Bảng điểm** 4 run: EM, ANLS, token-F1 (phụ), s/câu.
- **Hai phép so ghép cặp** bằng `compare_runs.py`: B1 vs B1+OCR, B2 vs B2+OCR (McNemar exact cho EM + CI95 bootstrap cho ANLS).
- **Kết luận theo luật:** "OCR cải thiện" chỉ khi EM tăng **và** p < 0,05.
- Ghi `ocr_ablation_summary.json` để Cursor đưa lên git.

*Kỳ vọng: bảng 4 dòng, 2 dòng `➜ …` (kết luận theo luật), file `ocr_ablation_summary.json`.*

In [ ]:
import pandas as pd
os.makedirs(OUT_DIR, exist_ok=True)
rows = []
for k, r in RUNS.items():
    m = json.load(open(f"{EXP_ROOT}/{r['run']}/metrics.json", encoding='utf-8'))
    mm = m['metrics']
    rows.append({'run': k, 'n': m['meta']['num_samples'], 'EM': round(mm['exact_match'], 4), 'ANLS': round(mm['anls'], 4),
                 'token-F1 (phụ)': round(mm.get('token_f1', float('nan')), 4), 's/câu': m['meta']['sec_per_sample']})
display(pd.DataFrame(rows))

summary = {'what': 'Ablation có / không OCR-enhanced prompting trên ViTextVQA test2000 (W08 bước 6)',
           'ocr_config_frozen_on_dev300': sel,
           'rule': 'OCR cải thiện nếu EM tăng và McNemar exact p < 0,05; ANLS kèm CI95 bootstrap ghép cặp (10.000, seed 42)',
           'table': rows, 'paired': {}, 'verdict': {}}
for base, ocr in (('B1', 'B1+OCR'), ('B2', 'B2+OCR')):
    out = f"{OUT_DIR}/paired_{base}_vs_{base}ocr.json"
    ra, rb = f"{EXP_ROOT}/{RUNS[base]['run']}", f"{EXP_ROOT}/{RUNS[ocr]['run']}"
    !python src/eval/compare_runs.py --run_a {ra} --run_b {rb} --label_a {base} --label_b {base}_OCR --out {out} > /dev/null
    p = json.load(open(out, encoding='utf-8'))
    p.pop('only_b_question_ids', None)
    summary['paired'][f'{base}_vs_{ocr}'] = p
    better = p['em_b'] > p['em_a'] and p['p_value'] < 0.05
    worse = p['em_b'] < p['em_a'] and p['p_value'] < 0.05
    v = 'CẢI THIỆN' if better else ('GIẢM có ý nghĩa' if worse else 'KHÔNG KHÁC BIỆT có ý nghĩa')
    summary['verdict'][ocr] = v
    print(f"➜ {ocr} so với {base}: EM {p['em_a']:.4f} -> {p['em_b']:.4f} | chỉ {base} đúng {p['n_only_a']}, "
          f"chỉ {ocr} đúng {p['n_only_b']}, p={p['p_value']:.3g} | ΔANLS {p['anls_diff']:+.4f} "
          f"CI95 [{p['anls_ci95'][0]:+.4f}; {p['anls_ci95'][1]:+.4f}] -> {v}")
json.dump(summary, open(f'{OUT_DIR}/ocr_ablation_summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('->', f'{OUT_DIR}/ocr_ablation_summary.json')

## 5. ⏸ DỪNG

Gửi Claude **output mục 4** (bảng + 2 dòng `➜`) và 2 dòng `== KẾT QUẢ` ở mục 3. Sau đó **Runtime → Disconnect and delete runtime**.

Bước tiếp theo là **B3 (QLoRA + OCR) và B2-sub (QLoRA, không OCR)** trên cùng tập con 8.000 câu. Đây là phép thử chính: train cùng OCR có dạy được model dùng OCR đúng cách hay không.